# Import libraries

In [ ]:
import os
import pandas as pd
from pathlib import Path
import platform
import requests
import sys
from time import sleep
from tqdm import tqdm
tqdm_bar_format = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"
import urllib.parse

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions
from dataverse_functions import dv_metrics_functions
from dataverse_functions import github_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

# Get info

- Import dataverse_installation_datacite_client_ids.csv as a dataframe
- Filter dataframe to only installations with DataCite client IDs in the relevant column
- For each remaining installation, get:
    - The count of datasets in DataCite with funderIdentifiers, using DataCite API
    - The count of datasets with ROR URL in Funding Information Agency field, using Dataverse API
    - If there's a difference between these two counts:
        - Create list 1, Using get_item_info_from_datacite_api() function to get DOIs and other get info about datasets in DataCite with funderIdenfifiers
        - Create list 2, using my function that uses Search API to get info about datasets, passing search URL that returns datasets with ROR URL in Funding Information Agency field
        - Get the list of dataset DOIs in list 1 that aren't in list 2
        - Get the list of dataset DOIs in list 2 that aren't in list 1

In [ ]:
datasetsWithFunderIdentifiersDf = pd.read_csv(os.path.join(desktopPath, 'dataverse_installation_datacite_client_ids.csv'))

with tqdm(bar_format=tqdm_bar_format, total=len(datasetsWithFunderIdentifiersDf)) as loopObj:
    for idx, installation in datasetsWithFunderIdentifiersDf.iterrows():
        loopObj.update(1)

        datacite_client_id = installation['datacite_client_id']
        if datacite_client_id is not None:

            # Get count of datasets in DataCite with funderIdentifiers
            funderIdentifiersInDataCite = f'https://api.datacite.org/dois?client-id={clientId}&state=findable&query=fundingReferences.funderIdentifier:*+NOT+relatedIdentifiers.relationType%3AIsPartOf'
            response = requests.get(funderIdentifiersInDataCite)
            funderIdentifiersInDataCiteDict = response.json()
            countOfDatasetsWithFunderIdentifiers = funderIdentifiersInDataCiteDict['meta']['total']

            # Get count of datasets in installation with ROR URL in Funding Information Agency field
            known_hostname = installation['known_hostname']
            installationUrl = installation['working_installation_url']
            rootCollectionInfoEndpoint = f'{installationUrl}/api/dataverses/:root'
            response = requests.get(rootCollectionInfoEndpoint, headers=headers, timeout=20, verify=False)
            sleep(1)
            rootCollectionJson = response.json()
            rootCollectionName = urllib.parse.quote(rootCollectionJson['data']['name'])
            funderRorSearchUrl = f'{installationUrl}/api/search?q=grantNumberAgency:"https://ror.org/"&fq=metadataSource:"{rootCollectionName}"&fq=publicationStatus:"Published"'
            response = requests.get(funderRorSearchUrl, headers=headers)
            countOfDatasetsWithFunderROR = response.json()... 
            sleep(1)

            if countOfDatasetsWithFunderIdentifiers != countOfDatasetsWithFunderROR:
                datasetInfoFromDataCiteDict = common_functions.get_item_info_from_datacite_api(funderIdentifiersInDataCite)
                datasetInfoFromDataCiteDf = pd.DataFrame(datasetInfoFromDataCiteDict)
                # Remove all columns except DOI column and columns about dataset creation and publication dates, versions, DataCite metadata schemaa, and funder info
                # Make sure DOI column is named "dataset_pid"
                # Add column to indicate that this info is from DataCite API
                
                datasetInfoFromDataverseDf = dataverse_functions.get_datasets_from_collection_or_search_url(funderRorSearchUrl)
                # Remove all columns except DOI column and columns about dataset creation and publication dates, versions, DataCite metadata schemaa, and funder info
                # Add column to indicate that this info is from Dataverse Search API

                # Merge both dataframes to create new dataframe with rows that aren't in both dataframes
    